In [1]:
!pip install datasets==3.6.0

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from typing import Tuple, Optional, Union

In [3]:
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Amazon_Fashion", trust_remote_code=True)
items = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Amazon_Fashion", split="full", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
reviews["full"]

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 2500939
})

In [5]:
items

Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
    num_rows: 826108
})

In [6]:
# Checking for items whose variant attribute is either not present or whose 1st item is not 'MAIN'
# filtered_items = items.filter(lambda x: x['images']['variant'] and x['images']['variant'][0] != 'MAIN')
# len(filtered_items)

In [7]:
# Checking for items that do not have a 'thumb' image or where 'thumb' is not a list or is an empty list
# no_thumb = items.filter(lambda x: 'thumb' not in x['images'] or not isinstance(x['images']['thumb'], list) or len(x['images']['thumb']) == 0)

In [8]:
# Convert both datasets to pandas DataFrames
main_reviews_df = reviews['full'].to_pandas()
main_items_df = items.to_pandas()

In [9]:
#Only keeping the verified reviews only!
main_reviews_df = main_reviews_df[main_reviews_df["verified_purchase"] == True]

In [64]:
######### NEED TO REMOVE DUPLICATESS #################

In [13]:
ts_num=pd.to_numeric(main_reviews_df["timestamp"], errors="coerce")
pd.to_datetime(ts_num, unit="ms", utc=True)

,timestamp
0,2020-01-09 00:06:34.489000+00:00
1,2020-12-20 01:04:06.701000+00:00
2,2015-05-23 01:33:48+00:00
3,2018-12-31 20:57:27.095000+00:00
4,2015-08-13 14:29:26+00:00
...,...
2500934,2016-06-24 20:12:38+00:00
2500935,2018-05-08 17:05:05.585000+00:00
2500936,2016-12-17 22:28:31+00:00
2500937,2017-04-15 17:34:26+00:00


In [14]:
TimestampLike = Union[pd.Timestamp, str, int, float]

def temporal_split_ms(
    df: pd.DataFrame,
    time_col: str,
    *,
    test_fraction: Optional[float] = None,   # exactly one of these
    cutoff: Optional[TimestampLike] = None,  # not both
    train_includes_cutoff: bool = True,
    drop_na_time: bool = True,
    sort_within_splits: bool = False,        # sort by the ms column itself
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    # ---- input checks
    if (test_fraction is None) == (cutoff is None):
        raise ValueError("Provide exactly one of `test_fraction` or `cutoff`.")
    if test_fraction is not None:
        if not np.isfinite(test_fraction):
            raise ValueError("`test_fraction` must be finite.")
        if not (0.0 < float(test_fraction) < 1.0):
            raise ValueError("`test_fraction` must be in (0, 1).")

    # ---- ensure numeric ms epoch
    ms = pd.to_numeric(df[time_col], errors="coerce")

    # drop invalid timestamps if requested
    if drop_na_time:
        valid = ms.notna()
        if not valid.all():
            df = df.loc[valid]
            ms = ms.loc[valid]
    if len(df) == 0:
        raise ValueError("All timestamps are NaN after parsing; nothing to split.")

    # ---- pick cutoff in **ms**
    if cutoff is None:
        q = 1.0 - float(test_fraction)
        cutoff_ms = ms.quantile(q, interpolation="nearest")
        if not np.isfinite(cutoff_ms):
            raise RuntimeError("Failed to compute a valid quantile cutoff.")
        cutoff_ms = int(cutoff_ms)
    else:
        if isinstance(cutoff, (int, float)) and np.isfinite(cutoff):
            cutoff_ms = int(cutoff)
        else:
            # parse to UTC and convert to ms
            ts = pd.to_datetime(cutoff, utc=True)
            if pd.isna(ts):
                raise ValueError("`cutoff` could not be parsed into a valid timestamp.")
            cutoff_ms = int(ts.value // 1_000_000)  # ns -> ms

    # sanity: cutoff within range
    mn, mx = ms.min(), ms.max()
    if not (mn <= cutoff_ms <= mx):
        raise RuntimeError(f"Cutoff {cutoff_ms} outside data range [{mn}, {mx}].")

    # ---- split using numeric masks (no datetime needed)
    if train_includes_cutoff:
        train_mask = ms.le(cutoff_ms)
        test_mask  = ms.gt(cutoff_ms)
    else:
        train_mask = ms.lt(cutoff_ms)
        test_mask  = ms.ge(cutoff_ms)

    train = df.loc[train_mask].copy()
    test  = df.loc[test_mask].copy()

    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(
            f"Empty split: train={len(train)}, test={len(test)}. "
            "Adjust `test_fraction` or `cutoff`."
        )
    if not train.index.intersection(test.index).empty:
        raise AssertionError("Split overlap detected (indices intersect).")

    # ---- optional: sort by the ms column itself (stable if you want ties preserved)
    if sort_within_splits:
        train = train.sort_values(time_col, kind="mergesort")
        test  = test.sort_values(time_col, kind="mergesort")

    # return cutoff as a human-readable Timestamp (naive UTC)
    cutoff_ts = pd.to_datetime(cutoff_ms, unit="ms", utc=True).tz_convert(None)
    return train, test, cutoff_ts


In [15]:
# 80/20 temporal split
train_df, test_df, cutoff_ts = temporal_split_ms(
    main_reviews_df, time_col="timestamp", test_fraction=0.2, sort_within_splits=True
)
print("Cutoff:", cutoff_ts, "Train:", len(train_df), "Test:", len(test_df))


Cutoff: 2020-12-31 19:23:33.359000 Train: 1870162 Test: 467540


In [47]:
def kcore_filter_iterative(
    df: pd.DataFrame,
    user_col: str = "user_id",
    item_col: str = "parent_asin",
    user_k: int = 5,
    item_k: int = 5,
    max_iters: int = 100,
    drop_duplicates: bool = False,        # drop exact duplicate (user,item,...) rows first
    return_history: bool = False,
    verbose: bool = False,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Iteratively prune users/items with degree < thresholds until convergence.
    Degrees are counted as row frequency (after optional de-duplication).
    Returns (filtered_df, history_df or None).
    """

    # ---- input checks
    if user_col not in df.columns or item_col not in df.columns:
        raise ValueError(f"Missing required columns: {user_col!r}, {item_col!r}")
    if not (isinstance(user_k, int) and user_k >= 1):
        raise ValueError("user_k must be an integer >= 1.")
    if not (isinstance(item_k, int) and item_k >= 1):
        raise ValueError("item_k must be an integer >= 1.")
    if not (isinstance(max_iters, int) and max_iters >= 1):
        raise ValueError("max_iters must be an integer >= 1.")

    cur = df
    if drop_duplicates:
        cur = cur.drop_duplicates(subset=[user_col, item_col], keep="first")

    if len(cur) == 0:
        raise ValueError("No rows to process after de-duplication (if enabled).")

    history = []
    prev_len = -1

    for it in range(1, max_iters + 1):
        n_before = len(cur)

        # prune users
        ucnt = cur[user_col].value_counts()
        cur = cur[cur[user_col].map(ucnt) >= user_k]
        if len(cur) == 0:
            raise RuntimeError(f"All rows pruned at user step (iter={it}). "
                               f"Consider lowering user_k/item_k.")

        # prune items (recompute counts after user pruning)
        icnt = cur[item_col].value_counts()
        cur = cur[cur[item_col].map(icnt) >= item_k]
        if len(cur) == 0:
            raise RuntimeError(f"All rows pruned at item step (iter={it}). "
                               f"Consider lowering user_k/item_k.")

        n_after = len(cur)
        step = {
            "iter": it,
            "rows_before": n_before,
            "rows_after": n_after,
            "users_after": cur[user_col].nunique(),
            "items_after": cur[item_col].nunique(),
            "removed": n_before - n_after,
        }
        history.append(step)
        if verbose:
            print(step)

        # convergence: no change
        if n_after == n_before or n_after == prev_len:
            break
        prev_len = n_after

    hist_df = pd.DataFrame(history)
    return (cur.reset_index(drop=True), hist_df if return_history else None)


In [59]:
# Example on train_df from your temporal split
warm_train_df, hist = kcore_filter_iterative(
    train_df,
    user_col="user_id",
    item_col="parent_asin",
    user_k=4,
    item_k=2,
    return_history=True,
    verbose=True
)

{'iter': 1, 'rows_before': 1850504, 'rows_after': 37343, 'users_after': 16159, 'items_after': 11746, 'removed': 1813161}
{'iter': 2, 'rows_before': 37343, 'rows_after': 8088, 'users_after': 2425, 'items_after': 2348, 'removed': 29255}
{'iter': 3, 'rows_before': 8088, 'rows_after': 3995, 'users_after': 939, 'items_after': 992, 'removed': 4093}
{'iter': 4, 'rows_before': 3995, 'rows_after': 2954, 'users_after': 605, 'items_after': 679, 'removed': 1041}
{'iter': 5, 'rows_before': 2954, 'rows_after': 2585, 'users_after': 503, 'items_after': 568, 'removed': 369}
{'iter': 6, 'rows_before': 2585, 'rows_after': 2419, 'users_after': 460, 'items_after': 525, 'removed': 166}
{'iter': 7, 'rows_before': 2419, 'rows_after': 2351, 'users_after': 443, 'items_after': 508, 'removed': 68}
{'iter': 8, 'rows_before': 2351, 'rows_after': 2329, 'users_after': 437, 'items_after': 504, 'removed': 22}
{'iter': 9, 'rows_before': 2329, 'rows_after': 2316, 'users_after': 434, 'items_after': 500, 'removed': 13}
{'i

In [60]:
merged_train_df = pd.merge(warm_train_df, main_items_df, on='parent_asin', how='left')

In [61]:
merged_train_df.shape

(2302, 25)

In [62]:
merged_train_df.duplicated(subset=['user_id', 'parent_asin']).sum()

np.int64(0)

In [63]:
merged_train_df["timestamp"]

,timestamp
0,1347420778000
1,1351210541000
2,1351210649000
3,1351212742000
4,1354641921000
...,...
2297,1579584725798
2298,1581085900142
2299,1585692925624
2300,1599164172226


In [65]:
def extract_main_image_url(image_dict):
    """
    Safely extracts the first valid (non-None) hi_res image URL from the image dictionary.
    """
    # Check if the image data and hi_res key are valid
    if isinstance(image_dict, dict) and 'hi_res' in image_dict and isinstance(image_dict['hi_res'], np.ndarray):
        # Iterate through the list of URLs
        for url in image_dict['hi_res']:
            # If the URL is a string (and not None), return it
            if isinstance(url, str):
                return url
    # If no valid URL was found after checking everything, return None
    return None

# Apply the corrected function
merged_train_df['main_image'] = merged_train_df['images_y'].apply(extract_main_image_url)

In [66]:
merged_train_df["main_image"]

,main_image
0,https://m.media-amazon.com/images/I/71e7WkcRQp...
1,https://m.media-amazon.com/images/I/51YVRMMK7L...
2,https://m.media-amazon.com/images/I/71lYjMM16S...
3,None
4,https://m.media-amazon.com/images/I/71lYjMM16S...
...,...
2297,https://m.media-amazon.com/images/I/611lH6Kj7+...
2298,https://m.media-amazon.com/images/I/71GFOKuvbh...
2299,https://m.media-amazon.com/images/I/61OTapGJEr...
2300,https://m.media-amazon.com/images/I/61kiAzBuBI...


In [67]:
# Now, drop only the rows where no valid image could be found at all
cleaned_df = merged_train_df.dropna(subset=['main_image'])

In [68]:
cleaned_df.shape

(2120, 26)